# Standard Deep Neural Network - Lung Cancer Dataset

This notebook trains a standard (non-private) DNN on the Lung Cancer dataset (50,000 samples, 23 features, 3-class target) to establish baseline performance.

## 1. Import Libraries

In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
import os
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    f1_score,
    precision_score,
    recall_score
)

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print(f"Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Shared metric collector (repository root): logs training accuracy,
# balanced accuracy and AUROC alongside the metrics already tracked below.
import os as _os
import sys as _sys
_REPO_ROOT = _os.path.abspath(_os.path.join('..', '..'))
if _REPO_ROOT not in _sys.path:
    _sys.path.insert(0, _REPO_ROOT)
from metrics_utils import ExtraMetrics


Using CPU
PyTorch version: 2.11.0
Device: cpu


## 2. Load Preprocessed Data

In [2]:
with open('../data/processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']
feature_names = data['feature_names']
n_classes = data.get('n_classes', 3)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Number of features: {X_train.shape[1]}")
print(f"Number of classes: {n_classes}")

X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)

batch_size = 256
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nData loaded and prepared")

Training set: (40000, 23)
Test set: (10000, 23)
Number of features: 23
Number of classes: 3

Data loaded and prepared


## 3. Define DNN Architecture

In [3]:
class DNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_sizes=[128, 64], num_classes=3, dropout=0.3):
        super(DNNClassifier, self).__init__()
        layers = []
        prev_size = input_size
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size
        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

input_size = X_train.shape[1]
hidden_sizes = [128, 64]
num_classes = n_classes
dropout = 0.3
learning_rate = 0.001
num_epochs = 50

print(f"Model Architecture:")
print(f"  Input: {input_size} features")
print(f"  Hidden layers: {hidden_sizes}")
print(f"  Output: {num_classes} classes")
print(f"  Dropout: {dropout}")

Model Architecture:
  Input: 23 features
  Hidden layers: [128, 64]
  Output: 3 classes
  Dropout: 0.3


## 4. Training Functions

In [4]:
def train_model(model, train_loader, criterion, optimizer, num_epochs, device, verbose=False):
    model.train()
    train_losses = []
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)
        if verbose and (epoch + 1) % 10 == 0:
            print(f"  Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")
    return train_losses

def evaluate_model(model, loader, device):
    """Return (labels, predictions, class probabilities) for `loader`.

    The probabilities are what AUROC needs; predictions and labels are
    unchanged from the original implementation.
    """
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            all_probs.extend(torch.softmax(outputs, dim=1).cpu().numpy())
            all_preds.extend(torch.max(outputs, 1)[1].cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

print("Training functions defined")

Training functions defined


## 5. Train Model (Multiple Runs)

In [5]:
N_RUNS = 10

detail_accuracies = []
detail_f1s = []
detail_precisions = []
detail_recalls = []
detail_cms = []

print(f"Training Standard DNN {N_RUNS} times...")
print('='*80)

extra = ExtraMetrics()
for run in range(N_RUNS):
    seed = run * 10 + 42
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    model = DNNClassifier(input_size, hidden_sizes, num_classes, dropout).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    _t0 = time.perf_counter()
    train_losses = train_model(model, train_loader, criterion, optimizer, num_epochs, device)
    train_time = time.perf_counter() - _t0
    # Training-partition score for the non-private fit.
    y_train_true, y_train_pred, _ = evaluate_model(model, train_loader, device)
    y_true, y_pred, y_prob = evaluate_model(model, test_loader, device)

    accuracy = accuracy_score(y_true, y_pred)
    extra.add(y_true, y_pred, y_score=y_prob, train_true=y_train_true, train_pred=y_train_pred, train_time=train_time)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    detail_accuracies.append(accuracy)
    detail_f1s.append(f1)
    detail_precisions.append(precision)
    detail_recalls.append(recall)
    detail_cms.append(cm)

    print(f"  Run {run+1:2d}/{N_RUNS} | Acc: {accuracy:.4f} | F1: {f1:.4f} | Prec: {precision:.4f} | Rec: {recall:.4f}")

print('='*80)
print('Training complete')

Training Standard DNN 10 times...


  Run  1/10 | Acc: 1.0000 | F1: 1.0000 | Prec: 1.0000 | Rec: 1.0000


  Run  2/10 | Acc: 1.0000 | F1: 1.0000 | Prec: 1.0000 | Rec: 1.0000


  Run  3/10 | Acc: 1.0000 | F1: 1.0000 | Prec: 1.0000 | Rec: 1.0000


  Run  4/10 | Acc: 1.0000 | F1: 1.0000 | Prec: 1.0000 | Rec: 1.0000


...

  Run  6/10 | Acc: 1.0000 | F1: 1.0000 | Prec: 1.0000 | Rec: 1.0000


  Run  7/10 | Acc: 1.0000 | F1: 1.0000 | Prec: 1.0000 | Rec: 1.0000


  Run  8/10 | Acc: 1.0000 | F1: 1.0000 | Prec: 1.0000 | Rec: 1.0000


  Run  9/10 | Acc: 1.0000 | F1: 1.0000 | Prec: 1.0000 | Rec: 1.0000


  Run 10/10 | Acc: 1.0000 | F1: 1.0000 | Prec: 1.0000 | Rec: 1.0000
Training complete


## 6. Aggregate Results

In [6]:
accuracy = float(np.mean(detail_accuracies))
f1 = float(np.mean(detail_f1s))
precision = float(np.mean(detail_precisions))
recall = float(np.mean(detail_recalls))
std_accuracy = float(np.std(detail_accuracies))
std_f1 = float(np.std(detail_f1s))

print('=' * 60)
print('STANDARD DNN - EVALUATION METRICS (LUNG CANCER DATASET)')
print('=' * 60)
print(f'Runs:      {N_RUNS}')
print(f'Accuracy:  {accuracy:.4f} +/- {std_accuracy:.4f}')
print(f'F1-Score:  {f1:.4f} +/- {std_f1:.4f} (weighted)')
print(f'Precision: {precision:.4f} (weighted)')
print(f'Recall:    {recall:.4f} (weighted)')
print('=' * 60)

STANDARD DNN - EVALUATION METRICS (LUNG CANCER DATASET)
Runs:      10
Accuracy:  1.0000 +/- 0.0000
F1-Score:  1.0000 +/- 0.0000 (weighted)
Precision: 1.0000 (weighted)
Recall:    1.0000 (weighted)


## 7. Save Baseline Results

In [7]:
os.makedirs('output', exist_ok=True)

baseline_results = {
    'model': 'Standard DNN',
    'dataset': 'Lung Cancer',
    'architecture': str(hidden_sizes),
    'input_features': input_size,
    'accuracy': accuracy,
    'f1_score': f1,
    'precision': precision,
    'recall': recall,
    'accuracy_std': std_accuracy,
    'f1_std': std_f1
}
baseline_results.update(extra.means())

baseline_df = pd.DataFrame([baseline_results])
baseline_df.to_csv('output/baseline_dnn_results.csv', index=False)

with open('output/std_dnn_report.json', 'w') as f:
    json.dump(baseline_results, f, indent=4)

print("Results saved to output/baseline_dnn_results.csv")

# Export model for MIA
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..', '..')))
from model_exporter import export_model
export_model(model, 'output/model/std_dnn_model.pt')


print("Added metrics:")
print(extra.describe())

Results saved to output/baseline_dnn_results.csv
PyTorch model successfully exported to output/model/std_dnn_model.pt
Added metrics:
  Train Acc:      1.0000 ± 0.0000
  Balanced Acc:   1.0000 ± 0.0000
  AUROC:          1.0000 ± 0.0000
  Train Time (s): 5.6571 ± 0.3259
